<div dir="rtl">

# NB01-M01 — Source Policy, Rights and Authority

## هدف
تعریف و اعتبارسنجی قرارداد رسمی منبع برای فاز یک QRAG.

وجود یک متن با پذیرفته‌شدن آن به‌عنوان منبع فعال یکی نیست. قرآن، ترجمه، تفسیر، حدیث، Claim استخراج‌شده و پاسخ مدل باید لایه‌های جدا داشته باشند.

### ورودی‌ها
- config/phase1/source_authority_policy_v1.json
- schemas/phase1/source_record.schema.json
- data/phase1/manifests/source_candidates_v1.json

### خروجی
- reports/phase1/source_policy_acceptance_v1.json

### خارج از Scope
دانلود تفسیر یا حدیث، پذیرش حقوقی قطعی، تعیین صحت حدیث توسط مدل و ورود Candidateها به Corpus فعال.

</div>

<div dir="rtl">

# NB01-M02 — اصول غیرقابل مذاکره

- قرآن عربی و ترجمه یک چیز نیستند.
- تفسیر باید به اثر و مفسر Attribution شود.
- حدیث Collection و Locator خود را حفظ می‌کند.
- درجه حدیث فقط با Source صریح نمایش داده می‌شود.
- اختلاف منابع با Merge خاموش حذف نمی‌شود.
- Claim باید به Evidence برگردد.
- پاسخ مدل Evidence نیست.
- حضور فایل در Repository مدرک حقوق استفاده نیست.

</div>

In [1]:
# NB01-C01 — Imports and project root discovery
import json
import sys
from pathlib import Path

current_directory = Path.cwd().resolve()
project_root = None

for candidate in [current_directory, *current_directory.parents]:
    if (
        (candidate / ".git").exists()
        and (candidate / "notebooks").exists()
        and (candidate / "data").exists()
    ):
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "QRAG project root could not be identified."
    )

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from qrag_phase1.foundation import (
    collect_git_info,
    sha256_file,
    write_json_atomic,
)

git_info = collect_git_info(project_root)
assert git_info["branch"] == "qrag-phase1"

print("Project root:", project_root)
print("Branch:", git_info["branch"])

Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/quran-rag
Branch: qrag-phase1


<div dir="rtl">

# NB01-M03 — Load Contracts

Policy، Schema و Candidate Manifest مستقیماً از Disk خوانده می‌شوند.

</div>

In [2]:
# NB01-C02 — Load policy, schema and candidate manifest
policy_path = (
    project_root
    / "config"
    / "phase1"
    / "source_authority_policy_v1.json"
)
schema_path = (
    project_root
    / "schemas"
    / "phase1"
    / "source_record.schema.json"
)
candidate_manifest_path = (
    project_root
    / "data"
    / "phase1"
    / "manifests"
    / "source_candidates_v1.json"
)

for path in (
    policy_path,
    schema_path,
    candidate_manifest_path,
):
    assert path.is_file(), f"Required contract file is missing: {path}"

with policy_path.open("r", encoding="utf-8") as handle:
    source_policy = json.load(handle)

with schema_path.open("r", encoding="utf-8") as handle:
    source_schema = json.load(handle)

with candidate_manifest_path.open("r", encoding="utf-8") as handle:
    candidate_manifest = json.load(handle)

print("Policy:", source_policy["policy_id"])
print("Candidate count:", len(candidate_manifest["candidates"]))

Policy: qrag:phase1:source-authority:v1
Candidate count: 14


<div dir="rtl">

# NB01-M04 — Policy Structure Validation

جداسازی Layerها و Ruleهای حیاتی بررسی می‌شوند.

</div>

In [3]:
# NB01-C03 — Validate policy structure
required_layers = {
    "quran_canonical_text",
    "translation",
    "tafsir",
    "hadith",
    "derived_claim",
    "generated_answer",
}

policy_layers = {
    item["id"]
    for item in source_policy["epistemic_layers"]
}

assert policy_layers == required_layers
assert "quran_only" in source_policy["answer_modes"]
assert "scholarly_synthesis" in source_policy["answer_modes"]
assert (
    source_policy["rights_policy"]["repository_presence_is_rights_evidence"]
    is False
)
assert (
    source_policy["rights_policy"]["unknown_or_unverified_action"]
    == "quarantine"
)
assert len(source_policy["hard_rules"]) >= 8

print("Source policy structure: PASS")

Source policy structure: PASS


<div dir="rtl">

# NB01-M05 — Candidate Manifest Validation

شناسه، Layer/Family و جلوگیری از پذیرش زودرس Sourceها بررسی می‌شود.

</div>

In [4]:
# NB01-C04 — Candidate manifest validation
candidates = candidate_manifest["candidates"]

source_ids = [
    item["source_id"]
    for item in candidates
]

assert len(source_ids) == len(set(source_ids))

family_to_layer = {
    "quran": "quran_canonical_text",
    "translation": "translation",
    "tafsir": "tafsir",
    "hadith": "hadith",
    "derived_claim": "derived_claim",
    "generated_answer": "generated_answer",
}

invalid_family_layer = []
premature_acceptance = []
invalid_external_paths = []

for item in candidates:
    expected_layer = family_to_layer[
        item["source_family"]
    ]

    if item["epistemic_layer"] != expected_layer:
        invalid_family_layer.append(item["source_id"])

    if (
        item["acceptance_status"] == "accepted"
        and item["rights_status"] != "verified_allowed"
    ):
        premature_acceptance.append(item["source_id"])

    if (
        item["acquisition_status"] == "not_acquired"
        and item["repository_path"] is not None
    ):
        invalid_external_paths.append(item["source_id"])

assert invalid_family_layer == []
assert premature_acceptance == []
assert invalid_external_paths == []

print("Candidate manifest structure: PASS")

Candidate manifest structure: PASS


<div dir="rtl">

# NB01-M06 — Repository-backed Candidate Check

فقط Candidateهایی که ادعا می‌کنند در Repository حاضرند روی Disk بررسی می‌شوند.

</div>

In [5]:
# NB01-C05 — Verify repository-backed candidates
repository_backed = [
    item
    for item in candidates
    if item["acquisition_status"] == "present_in_repository"
]

missing_repository_candidates = []

for item in repository_backed:
    relative_path = item["repository_path"]
    file_path = project_root / relative_path

    if not file_path.is_file():
        missing_repository_candidates.append(
            item["source_id"]
        )

assert missing_repository_candidates == []

print(
    "Repository-backed candidates:",
    len(repository_backed),
)

Repository-backed candidates: 3


<div dir="rtl">

# NB01-M07 — اولویت منابع شیعی بدون پذیرش خودکار

اولویت شیعی Discovery را هدایت می‌کند ولی هیچ Candidate را خودکار Accepted نمی‌کند.

</div>

In [6]:
# NB01-C06 — Validate tradition-priority contract
assert (
    source_policy["tradition_scope"]["tafsir_and_hadith_priority"]
    == "shia_priority"
)

tafsir_candidates = [
    item
    for item in candidates
    if item["source_family"] == "tafsir"
]

hadith_candidates = [
    item
    for item in candidates
    if item["source_family"] == "hadith"
]

assert tafsir_candidates
assert hadith_candidates

for item in tafsir_candidates + hadith_candidates:
    assert item["acceptance_status"] != "accepted"
    assert item["rights_status"] in {
        "unknown",
        "evidence_required",
        "restricted",
        "prohibited",
    }

print("Tafsir candidates:", len(tafsir_candidates))
print("Hadith candidates:", len(hadith_candidates))

Tafsir candidates: 5
Hadith candidates: 6


<div dir="rtl">

# NB01-M08 — Citation Contract Validation

حداقل Locator هر خانواده از ابتدا تعریف می‌شود.

</div>

In [7]:
# NB01-C07 — Validate citation contract
citation_policy = source_policy["citation_policy"]

assert citation_policy["quran"] == [
    "surah",
    "ayah",
]

assert "translation_id" in citation_policy["translation"]
assert "work_id" in citation_policy["tafsir"]
assert "stable_locator" in citation_policy["tafsir"]
assert "collection_id" in citation_policy["hadith"]
assert (
    "hadith_id_or_stable_locator"
    in citation_policy["hadith"]
)

print("Citation contract: PASS")

Citation contract: PASS


<div dir="rtl">

# NB01-M09 — Conflict Preservation Contract

Agreement و Difference از ابتدا Relation مستقل دارند.

</div>

In [8]:
# NB01-C08 — Validate conflict relation contract
required_relations = {
    "AGREES_WITH",
    "COMPLEMENTS",
    "PARTIAL_OVERLAP",
    "DIFFERS_FROM",
    "CONFLICTS_WITH",
    "UNRESOLVED",
}

assert set(
    source_policy["conflict_relation_types"]
) == required_relations

print("Conflict-preservation contract: PASS")

Conflict-preservation contract: PASS


<div dir="rtl">

# NB01-M10 — Build Acceptance Report

Acceptance این Notebook فقط Policy را می‌پذیرد، نه Source Candidateهای خارجی را.

</div>

In [9]:
# NB01-C09 — Build source policy acceptance report
reports_directory = project_root / "reports" / "phase1"
reports_directory.mkdir(parents=True, exist_ok=True)

external_candidates = [
    item
    for item in candidates
    if item["acquisition_status"] == "not_acquired"
]

rights_unverified = [
    item["source_id"]
    for item in candidates
    if item["rights_status"] != "verified_allowed"
]

active_candidates = [
    item["source_id"]
    for item in candidates
    if item["acceptance_status"] == "accepted"
]

failed_checks = []
warnings = []

if active_candidates:
    failed_checks.append(
        "CANDIDATE_MANIFEST_CONTAINS_ACTIVE_SOURCES"
    )

if external_candidates:
    warnings.append(
        f"EXTERNAL_CANDIDATES_REQUIRE_ACQUISITION:{len(external_candidates)}"
    )

if rights_unverified:
    warnings.append(
        f"SOURCES_REQUIRE_RIGHTS_EVIDENCE:{len(rights_unverified)}"
    )

acceptance_status = (
    "FAIL"
    if failed_checks
    else "PASS_WITH_LIMITATIONS"
)

acceptance_report = {
    "schema_version": "qrag_nb01_acceptance_v1",
    "objective_status": (
        "ACHIEVED" if not failed_checks else "NOT_ACHIEVED"
    ),
    "acceptance_status": acceptance_status,
    "failed_checks": failed_checks,
    "warnings": warnings,
    "policy_path": str(
        policy_path.relative_to(project_root)
    ),
    "policy_sha256": sha256_file(policy_path),
    "schema_path": str(
        schema_path.relative_to(project_root)
    ),
    "schema_sha256": sha256_file(schema_path),
    "candidate_manifest_path": str(
        candidate_manifest_path.relative_to(project_root)
    ),
    "candidate_manifest_sha256": sha256_file(
        candidate_manifest_path
    ),
    "candidate_count": len(candidates),
    "repository_backed_candidate_count": len(repository_backed),
    "external_candidate_count": len(external_candidates),
    "accepted_source_count": len(active_candidates),
    "rights_unverified_count": len(rights_unverified),
    "policy_decision": (
        "Policy accepted. Candidate sources remain non-active "
        "until source-specific inventory and acceptance notebooks."
    ),
    "next_notebook": (
        "notebooks/phase1/02_quran_canonical_inventory.ipynb"
        if not failed_checks
        else None
    ),
}

acceptance_report_path = (
    reports_directory / "source_policy_acceptance_v1.json"
)

write_json_atomic(
    acceptance_report_path,
    acceptance_report,
)

with acceptance_report_path.open(
    "r",
    encoding="utf-8",
) as handle:
    acceptance_reload = json.load(handle)

assert acceptance_reload == acceptance_report

print(json.dumps(
    acceptance_report,
    ensure_ascii=False,
    indent=2,
))

{
  "schema_version": "qrag_nb01_acceptance_v1",
  "objective_status": "ACHIEVED",
  "acceptance_status": "PASS_WITH_LIMITATIONS",
  "failed_checks": [],
  "warnings": [
    "EXTERNAL_CANDIDATES_REQUIRE_ACQUISITION:11",
    "SOURCES_REQUIRE_RIGHTS_EVIDENCE:13"
  ],
  "policy_path": "config/phase1/source_authority_policy_v1.json",
  "policy_sha256": "d2270bea6c97c0ea30a628c51db5b0cdb7377391a163831a6f14d13808803373",
  "schema_path": "schemas/phase1/source_record.schema.json",
  "schema_sha256": "652c1295ddcfcd0274a95c5bea3c47be1dc64c2ebc2cf9de0c4ccb1798a079f5",
  "candidate_manifest_path": "data/phase1/manifests/source_candidates_v1.json",
  "candidate_manifest_sha256": "b0ffa525b047277e71322571b7711b8593dd954bb851cdaf0bf844ea199637e2",
  "candidate_count": 14,
  "repository_backed_candidate_count": 3,
  "external_candidate_count": 11,
  "accepted_source_count": 0,
  "rights_unverified_count": 13,
  "policy_decision": "Policy accepted. Candidate sources remain non-active until source-sp

<div dir="rtl">

# NB01-M11 — Final Health Gate

NB02 فقط Inventory و Canonical audit قرآن موجود را شروع می‌کند؛ هیچ Tafsir یا Hadith هنوز Ingest نمی‌شود.

</div>

In [10]:
# NB01-C10 — Final health gate
assert acceptance_report["failed_checks"] == []
assert acceptance_report["accepted_source_count"] == 0
assert acceptance_report["candidate_count"] == len(candidates)
assert acceptance_report["next_notebook"].endswith(
    "02_quran_canonical_inventory.ipynb"
)

print(
    "NB01 objective: ACHIEVED "
    "(policy accepted, sources not yet activated)"
)

NB01 objective: ACHIEVED (policy accepted, sources not yet activated)


<div dir="rtl">

# NB01-M12 — جمع‌بندی

از این مرحله به بعد هیچ Source بدون هویت، Edition، Locator، Provenance و وضعیت استفاده وارد Corpus فعال نمی‌شود. اولویت Tafsir و Hadith شیعی است، اما پذیرش هر اثر باید در Notebookهای خودش با Evidence انجام شود.

تا قبل از اجرای واقعی Restart Kernel → Run All وضعیت PENDING_LOCAL_RUN_ALL باقی می‌ماند.

</div>